# Module 39.2 — Feature Extraction


---

## 1. What is Feature Extraction?

Feature extraction is a transfer learning technique in which we use a pretrained neural network as a fixed feature extractor.

We reuse the pretrained backbone without changing its learned weights and train a new classification head for our target dataset.

Let's use our industrial motor defect classification example.

Suppose we have four classes:

- Normal
- Crack
- Surface defect
- Deformation

Instead of training a CNN from scratch, we use a pretrained ResNet18 model and adapt its output to these four classes.

### Feature extraction workflow

```text
Pretrained Backbone
       ↓
Freeze Backbone
       ↓
New Classification Head
       ↓
Train
       ↓
Predicted Classes
```

![Transfer learning for industrial fault diagnosis](https://images.openai.com/static-rsc-4/evSo_JDgwEK-XHBUcxjhp5tzdnO7bKFatkxU3mb7iyXY0DwNu7_dnhRNg5t_o8oSt3dViDrtxJjsKEa3RyalUutfiBs25PUX3MnJxBSQ01I-wTAMdDbnqqZEvGf8IM34wbWlHKpjuLrv-n1M9ucIoOr29Z_SlyYitTfUXA9q3kk?purpose=inline)

[Source: MDPI — Intelligent Fault Diagnosis of Industrial Bearings Using Transfer Learning](https://www.mdpi.com/1424-8220/23/1/211)

**Central idea:** The backbone already knows how to extract useful visual features. We only teach a new classifier how to interpret those features for our problem.

---

## 2. Understand Each Stage

### Step 1 — Pretrained Backbone

A backbone is the main feature-extracting part of a neural network.

Examples:

- ResNet18
- ResNet50
- VGG16
- EfficientNet
- MobileNet

![Deep learning models and transfer learning](https://images.openai.com/static-rsc-4/LNl__SakmG2Yp3xiKVEaAz97OGTsTou344eVQ8v6eTsJethloM8s3fLYnKSc1b-AZNDeQf8Dtd3gTiBF9E3Z-yejhMLAALQTQI6Hk0gg9Dtgioh1uomNObcbVrT0P8IK0QzKrHAXpz7GKagpRo5EcSKth7DeQ85f6np5HKqSIHU?purpose=inline)

[Source: Appsilon — FastAI in R](https://www.appsilon.com/post/fastai-in-r)

A pretrained ResNet18 has already learned visual representations from ImageNet.

When an image passes through it, the backbone converts the image into a numerical feature representation.

For ResNet18, the feature vector before the original classification layer contains 512 values.

Conceptually:

```text
Input Image
    ↓
ResNet18 Backbone
    ↓
512-dimensional feature vector
```

These 512 values encode learned visual information about the image.

### Step 2 — Freeze Backbone

Freezing means preventing the pretrained backbone's parameters from being updated during training.

**Why?**

Because we want to preserve the useful visual features that the model has already learned.

In PyTorch:

```python
for param in model.parameters():
    param.requires_grad = False
```

Now, the backbone's parameters will not receive gradient-based updates.

**Important distinction:**

- Frozen parameters: Their weights do not change.
- Trainable parameters: Their weights can be updated by the optimizer.

### Step 3 — New Classification Head

The original ResNet18 classification layer predicts 1,000 ImageNet classes.

But our target problem has only four classes.

Therefore, we replace the original classification head.

```python
import torch.nn as nn

model.fc = nn.Linear(
    model.fc.in_features,
    4
)
```

Architecture:

```text
ResNet18 Backbone
       ↓
512 Features
       ↓
Linear Layer
       ↓
4 Logits
       ↓
Normal / Crack / Surface Defect / Deformation
```

![Deep learning classification architecture](https://images.openai.com/static-rsc-4/w-qxJjPpVothZej91SS80AdvYevZ1CbY2XPBRUP7okSospC0F-z0C6YBgRBwI4ipZXm3clNxo09TXdgBxRk4tM1r7AhNx22ZF60TYIdo1HfmIAzpGCWQRcNloxqrTxqtXBIBLjlGQLpaP015O5eppgrSvsSzHfwy4Xz5w39ef1I?purpose=inline)

[Source: MDPI — Multi-Criteria Spare Parts Classification Using Deep CNN](https://www.mdpi.com/2076-3417/11/15/7088)

### Step 4 — Train

Now we train only the new classification head.

The backbone extracts features, and the classification head learns the relationship between those features and our target labels.

For example:

```text
Input: Motor component image

Extracted features:
[0.21, 0.87, 0.34, ... 512 values]

Classification head output:
[2.1, 0.4, 3.8, 0.2]

Predicted class:
Surface defect
```

The values above are illustrative.

---

## 3. Complete PyTorch Implementation

Let's implement feature extraction using pretrained ResNet18.

Our target task has four classes:

```python
classes = [
    "normal",
    "crack",
    "surface_defect",
    "deformation"
]
```

### Step 1 — Load the pretrained model

```python
import torch
import torch.nn as nn
from torchvision import models

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

print("Device:", device)
```

### Step 2 — Freeze the backbone

```python
for param in model.parameters():
    param.requires_grad = False
```

All existing ResNet18 parameters are now frozen.

### Step 3 — Replace the classification head

```python
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    len(classes)
)

model = model.to(device)
```

**Why use `model.fc.in_features` instead of directly writing `512`?**

Because it retrieves the expected input dimension from the existing model architecture. This makes the code easier to adapt to other models.

### Step 4 — Configure the loss and optimizer

```python
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)
```

Notice this:

```python
optimizer = torch.optim.Adam(
    model.fc.parameters()
)
```

We pass only the classification head's parameters to the optimizer.

Therefore, only the new classifier gets updated.

### Step 5 — Train the model

Assuming `train_loader` and `val_loader` are already prepared:

```python
epochs = 10

for epoch in range(epochs):

    model.eval()
    model.fc.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    accuracy = 100 * correct / total
```

A subtle but important PyTorch detail: We use `model.eval()` and then `model.fc.train()` here. This keeps the frozen backbone's BatchNorm layers in evaluation mode, preventing their running statistics from changing, while still allowing gradients to update the classifier. `eval()` does not disable gradients.

### Step 6 — Verify trainable parameters

```python
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
```

For ResNet18 with a four-class linear head, the new head has:

$$
512 \times 4 + 4 = 2052
$$

Only **2,052 parameters** are trainable.

The original ResNet18 contains approximately 11.7 million parameters, so the vast majority remain frozen.

---

## 4. What Happens During Feature Extraction?

![Feature extraction in deep learning](https://images.openai.com/static-rsc-4/45cyuOxlw2WiAwQLkMcrx9Z7XnMDVJjDFYODLm-_eJlVr9_pvyCrsz9mg30ZYgD5p5rDPORCdTd9a5udCaqZjiTOuHKCJHMqnas2e3OIlVeRBgEF0ESrEP7zVU2-9UmfnzGmzeFSWaztt25seYgDSV5hSw4l7AeqONUADxQIMMs?purpose=inline)

[Source: Deep Learning — Feature Extraction](https://perso.esiee.fr/~chierchg/deep-learning/tutorials/transfer/transfer-2.html)

Let's understand the actual data flow.

| Stage | Input/output shape |
|---|---|
| Input image | `[32, 3, 224, 224]` |
| ResNet18 backbone | Feature extraction |
| Feature representation | `[32, 512]` |
| New classification head | `[32, 4]` |
| Predicted classes | `[32]` |

Here, the batch size is 32 and the model predicts four classes.

### Important distinction: Feature extraction vs. fine-tuning

![Transfer learning comparison for computer vision](https://images.openai.com/static-rsc-4/iG7RmsvzwLWlvozhzkeiG70OaOY1EWqpnQElAQUhp3AFV9PFHkvdwmxdDdZylc3kq82z3LWMyfb1U_S0GWt8fbmjbA8t3htUl8a7leSxka-9q1U37rTLyuIrCpcELWqAVPkm_ReVKRgQqKcqgJGPLBckTcwCfGM3a-roMlQSbLs?purpose=inline)

[Source: MDPI — Comparison of Deep Neural Networks in Detecting Field Grapevine Diseases](https://www.mdpi.com/2072-4292/14/18/4648)

- **Feature extraction:** Backbone frozen, classifier trainable.
- **Fine-tuning:** Selected backbone layers and classifier trainable.

The PyTorch tutorial demonstrates both strategies using ResNet18.

<row>
  <favicon url="https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html" size="24px"/>
  [PyTorch — Transfer Learning Tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
</row>

---

## 5. Advantages and Limitations

| Advantages | Limitations |
|---|---|
| Faster training | Limited backbone adaptation |
| Fewer trainable parameters | Pretrained features may not suit the target domain |
| Lower gradient-computation memory | May underperform fine-tuning |
| Useful for small datasets | Depends on source-model representations |

**Industrial engineering insight:** If your motor defect images are very different from ImageNet photographs, the pretrained backbone may not extract sufficiently specialised features. That is when fine-tuning becomes worth experimenting with.

---

## 6. What You Should Remember 🧠

1. Feature extraction reuses a pretrained CNN as a fixed feature extractor.
2. Freezing prevents backbone parameters from receiving gradient updates.
3. The original classification head is replaced with a new target-specific head.
4. Only the new classification head is trained.
5. The optimizer should receive only trainable parameters.
6. Frozen BatchNorm layers should be handled carefully during training.
7. Feature extraction is a useful baseline before attempting fine-tuning.

**One-line understanding:**

> The pretrained backbone extracts features; the new classification head learns how to classify them.

---

## 7. Official Learning Resources

- [PyTorch — Transfer Learning for Computer Vision](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial)
- [PyTorch — Official Implementation Source Code](https://github.com/pytorch/tutorials/blob/main/beginner_source/transfer_learning_tutorial.py)
- [TensorFlow — Feature Extraction Tutorial](https://www.tensorflow.org/tutorials/images/transfer_learning)

---